### Make sure you use REGF1W_ID

In [ ]:
import andes
import numpy as np
import pandas as pd
import os
import random

import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

import scipy.io as sio

MODEL = 'ANDES_Dutch_2035_mpc_ID.xlsx'

In [2]:
ss = andes.prepare(models=['REGF1W_MPC_ID'])

Generating code for 1 models on 6 processes.


### Time Domain Simulation

In [4]:
def gen_id_data(inv_or_loc_or_seed, size, u_or_d, extra_inv):
    dt = 0.01
    t_event = 10
    t_fin = 30

    ss = andes.main.load(MODEL, setup=False, config_path = "config_file.rc")
    if u_or_d == 'u_step':
        for i in range(8):
            ss.REGF1W_MPC_ID.PRBS.v[i] = 0
        ss.setup()
        ss.PFlow.run()
        ss.TDS.config.tstep = dt
        ss.TDS.config.tf = t_event
        ss.TDS.run()
        ss.REGF1W_MPC_ID.P_ID.v[inv_or_loc_or_seed] += size
        ss.TDS.config.tf = t_fin
        ss.TDS.run()
    elif u_or_d == 'd':
        for i in range(8):
            ss.REGF1W_MPC_ID.PRBS.v[i] = 0
        ss.setup()
        ss.PFlow.run()
        ss.TDS.config.tstep = dt
        ss.TDS.config.tf = t_event
        ss.TDS.run()
        ss.PQ.Ppf.v[inv_or_loc_or_seed] += size
        ss.TDS.config.tf = t_fin
        ss.TDS.run()
    elif u_or_d == 'du':
        for i in range(8):
            ss.REGF1W_MPC_ID.PRBS.v[i] = 0
        ss.setup()
        ss.PFlow.run()
        ss.TDS.config.tstep = dt
        ss.TDS.config.tf = t_event
        ss.TDS.run()
        ss.PQ.Ppf.v[inv_or_loc_or_seed] += size
        ss.TDS.config.tf = 20
        ss.TDS.run()
        ss.REGF1W_MPC_ID.P_ID.v[extra_inv] += 0.03
        ss.TDS.config.tf = t_fin
        ss.TDS.run()
    elif u_or_d == 'u_prbs':
        for i in range(8):
            ss.REGF1W_MPC_ID.PRBS.v[i] = 1
            ss.REGF1W_MPC_ID.seed.v[i] = inv_or_loc_or_seed
        ss.setup()
        ss.PFlow.run()
        ss.TDS.config.tstep = dt
        ss.TDS.config.tf = t_fin
        ss.TDS.run()
    elif u_or_d == 'du_prbs':
        for i in range(8):
            ss.REGF1W_MPC_ID.PRBS.v[i] = 1
            ss.REGF1W_MPC_ID.seed.v[i] = inv_or_loc_or_seed
        event_times = np.linspace(t_event,10*int(t_fin*0.1),int(t_fin*0.1)) #np.linspace(t_event,5*int(t_fin*0.2)-5,int(t_fin*0.2)-2)
        ss.setup()
        ss.PFlow.run()
        ss.TDS.config.tstep = dt
        random_events = {}
        random.seed(inv_or_loc_or_seed)
        Pload_0 = 8*[0]
        for i,event_t in enumerate(event_times):
            ss.TDS.config.tf = event_t
            ss.TDS.run()
            #sizes = [0 for _ in range(8)]
            #sizes[i] = random.choice([0.3,-0.3])
            if i == 0:
                Pload_0[0] = ss.PQ.Ppf.v[21]
                Pload_0[1] = ss.PQ.Ppf.v[8]
                Pload_0[2] = ss.PQ.Ppf.v[16]
                Pload_0[3] = ss.PQ.Ppf.v[5]
                Pload_0[4] = ss.PQ.Ppf.v[20]
                Pload_0[5] = ss.PQ.Ppf.v[9]
                Pload_0[6] = ss.PQ.Ppf.v[17]
                Pload_0[7] = ss.PQ.Ppf.v[0]
            sizes = [random.choice([0.2, 0.0]) for _ in range(8)]
            random_events[event_t] = sizes
            ss.PQ.Ppf.v[21] = Pload_0[0] +  sizes[0]
            ss.PQ.Ppf.v[8] = Pload_0[1] +sizes[1]
            ss.PQ.Ppf.v[16] = Pload_0[2] +sizes[2]
            ss.PQ.Ppf.v[5] = Pload_0[3] +sizes[3]
            ss.PQ.Ppf.v[20] = Pload_0[4] +sizes[4]
            ss.PQ.Ppf.v[9] = Pload_0[5] +sizes[5]
            ss.PQ.Ppf.v[17] = Pload_0[6] +sizes[6]
            ss.PQ.Ppf.v[0] = Pload_0[7] +sizes[7]
        ss.TDS.config.tf = t_fin
        ss.TDS.run()
    else:
        print(f'{u_or_d} should be a string u_step, u_prbs, d, du or du_prbs')
    
    df = ss.dae.ts.df
    time = df.index
    gen_df = pd.read_excel(MODEL, sheet_name='GENROU')
    f_df = df.filter(regex=r"^f BusROCOF")
    bus_numbers = f_df.columns.str.extract(r'(\d+)').astype(int)[0]
    gen_df["weight"] = gen_df["Sn"] * (gen_df["M"]/2)
    bus_weights = gen_df.groupby("bus")["weight"].sum()
    weights = bus_numbers.map(bus_weights).fillna(0.0).to_numpy()
    f_COI = (f_df.to_numpy() @ weights) / weights.sum()

    ids = [43, 44, 52, 53, 54, 55, 56, 57]
    bus_map = {43: 28, 44: 28, 52: 28, 53: 11, 54: 28, 55: 22, 56: 22, 57: 21,}

    t = np.around(np.asarray(time), 3)
    mask = ~pd.Series(t).duplicated()

    data = {}
    data['h'] = dt
    data['t'] = t[mask].reshape(-1,1)

    load_map = {21: 1, 8: 2, 16: 3, 5: 4, 20: 5, 9: 6, 17: 7, 0: 8}
    d_0 = np.zeros_like(time)
    d = np.zeros_like(time)
    if u_or_d == 'd' or u_or_d == 'du':
        which_d = load_map[inv_or_loc_or_seed]
        d[time >= t_event] = size
        data[f'd_{which_d}'] = d[mask].reshape(-1,1)
        for i in range(8):
            if i+1 !=which_d:
                data[f'd_{i+1}'] = d_0[mask].reshape(-1,1)
    elif u_or_d == 'du_prbs':
        d_sigs = [np.zeros_like(time) for _ in range(8)]
        for t_event in event_times:
            sizes = random_events[t_event]
            for i in range(8):
                d_sigs[i][time >= t_event] = sizes[i]
        for i in range(8):
            data[f'd_{i+1}'] = d_sigs[i][mask].reshape(-1,1)
    else:
        for i in range(8):
            data[f'd_{i+1}'] = d_0[mask].reshape(-1,1)

    
    u_df = df.filter(regex=r"^P_id REGF1W MPC ID")
    P_df = df.filter(regex=r"^Pe REGF1W MPC ID")

    data['f_COI'] = (50 * f_COI - 50)[mask].reshape(-1,1)

    for i in ids:
        data[f'Pmpc_{i}'] = u_df[f'P_id REGF1W MPC ID {i}'].to_numpy()[mask].reshape(-1,1)
        data[f'f_{i}']    = (50 * df[f'f BusROCOF {bus_map[i]}'] - 50).to_numpy()[mask].reshape(-1,1)
        data[f'Pe_{i}']   = (P_df[f'Pe REGF1W MPC ID {i}'] - P_df[f'Pe REGF1W MPC ID {i}'].iloc[0]).to_numpy()[mask].reshape(-1,1)
    
    for i,col in enumerate(f_df.columns):
        data[f'f_bus_{i}']    = (50 * df[col] - 50).to_numpy()[mask].reshape(-1,1)
    


    if u_or_d == 'u_step':
        inv_ls = [43, 44, 52, 53, 54, 55, 56, 57]
        inv = inv_ls[inv_or_loc_or_seed]
        sio.savemat(f'id_data/IBR={inv}_u={size}.mat', data)
    elif u_or_d == 'd':
        sio.savemat(f'id_data/loc={inv_or_loc_or_seed}_d={size}.mat', data)
    elif u_or_d == 'u_prbs':
        sio.savemat(f'id_data/u=PRBS_{int(inv_or_loc_or_seed)}.mat', data)
    elif u_or_d == 'du_prbs':
        sio.savemat(f'id_data/du=PRBS_{int(inv_or_loc_or_seed)}.mat', data)
    elif u_or_d == 'du':
        inv_ls = [43, 44, 52, 53, 54, 55, 56, 57]
        sio.savemat(f'id_data/IBR={inv_ls[extra_inv]}_loc={inv_or_loc_or_seed}.mat', data)

    
    return

invs = [0, 1, 2, 3, 4, 5, 6, 7]
locs = [21,8,16,5,20,9,17,0]
u_seeds = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
d_seeds = [4,5,6,7,8,9,10]
experiments = []

#for inv in invs:
#    experiments = experiments + [[inv,-0.02,'u_step',0]]
#    experiments = experiments + [[inv,-0.03,'u_step',0]]
#    experiments = experiments + [[inv,-0.04,'u_step',0]]
#for loc in locs:
#    experiments = experiments + [[loc,0.2,'d',0]]
#    experiments = experiments + [[loc,0.3,'d',0]]
#for seed in u_seeds:
#     experiments = experiments + [[seed,0,'u_prbs',0]]
#for seed in d_seeds:
#     experiments = experiments + [[seed,0,'du_prbs',0]]

for inv in invs:
    for loc in locs:
        experiments = experiments + [[loc,0.3,'du',inv]]

for exp in experiments:
    gen_id_data(exp[0], exp[1], exp[2], exp[3])
    print(f'Finished with exp = {exp}')

  0%|                                                    | 0/100 [00:00<?, ?%/s]

  0%|                                                    | 0/100 [00:00<?, ?%/s]

  0%|                                                    | 0/100 [00:00<?, ?%/s]

Finished with exp = [21, 0.3, 'du', 0]


  0%|                                                    | 0/100 [00:00<?, ?%/s]

...

Finished with exp = [17, 0.3, 'du', 7]


  0%|                                                    | 0/100 [00:00<?, ?%/s]

  0%|                                                    | 0/100 [00:00<?, ?%/s]

  0%|                                                    | 0/100 [00:00<?, ?%/s]

Finished with exp = [0, 0.3, 'du', 7]
